[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/duoan/TorchCode/blob/master/templates/20_weight_init.ipynb)

# 🟢 Easy: Kaiming Initialization

Implement **Kaiming (He) normal initialization** for weight tensors.

$$W \sim \mathcal{N}(0, \text{std}^2) \quad \text{where} \quad \text{std} = \sqrt{\frac{2}{\text{fan\_in}}}$$

### Signature
```python
def kaiming_init(weight: Tensor) -> Tensor:
    # Initialize weight in-place with Kaiming normal
    # fan_in = weight.shape[1]
    # Returns the weight tensor
```

#### Kaiming Initializationとは
ニューラルネットの**重みの初期値**を決める手法の一つ。
**He Initialization**ともよばれる(提唱者 He Kaiming の名前)。\
重みを平均0・標準偏差$\sqrt{2/\text{fan\_in}}$の正規分布からサンプリングする：
$$
W \sim \mathcal{N}(0, \text{std}^2) \quad \text{where} \quad \text{std} = \sqrt{\frac{2}{\text{fan\_in}}}
$$
`fan_in`はその層への入力数(前の層のニューロン数)

#### なぜ単純なrandnではダメか
問題18ではEmbeddingを`torch.randn`(標準偏差1)で初期化した。しかし層を重ねると、各層で値のばらつき(分散)が掛け算的に増減し、
- 深い層で信号が発散(値が爆発) $\rightarrow$ 勾配爆発
- または消失(値が0に潰れる) $\rightarrow$ 勾配消失
どちらも学習が進まなくなる。`fan_in`に応じて標準偏差を調整することで、**層を通っても分散が一定に保たれる**ようにするのがKaiming初期化。

#### なぜ分子が「2」か
ReLU(and/or GELU)は入力の負側を捨てるため、出力の分散がおよそ半分になる。これを補償するために分子を2にして標準偏差を$\sqrt{2}$倍している。
- 分子2 (std=$\sqrt{2/\text{fan\_in}}$) $\rightarrow$ **ReLU系活性化向け = Kaiming初期化**
- 分子1 (std=$\sqrt{1/\text{fan\_in}}$) $\rightarrow$ **tanh/sigmoid向け = Xavier初期化**

#### Deep Learningでの使われ方
**ReLU系活性化を使う層の標準的な初期化手法**。PyTorchの`nn.Linear`や`nn.Conv2d`の内部初期化でも(Kaiming系が)使われている。CNNや深いネットワークを安定して学習させるための基礎技術で、現代のモデルでは初期化を意識せずにつかえるのは、この種の手法がフレームワークに組み込まれている為。

In [1]:
# Install torch-judge in Colab (no-op in JupyterLab/Docker)
try:
    import google.colab
    get_ipython().run_line_magic('pip', 'install -q torch-judge')
except ImportError:
    pass


In [2]:
import torch
import math

/usr/local/lib/python3.11/site-packages/torch/_subclasses/functional_tensor.py:307: UserWarning: Failed to initialize NumPy: No module named 'numpy' (Triggered internally at /pytorch/torch/csrc/utils/tensor_numpy.cpp:84.)
  cpu = _conversion_method_template(device=torch.device("cpu"))


In [13]:
# ✏️ YOUR IMPLEMENTATION HERE

def kaiming_init(weight):
    fan_in = weight.shape[1] if weight.dim() >= 2 else weight.shape[0]
    std = math.sqrt(2.0 / fan_in)
    with torch.no_grad(): # 重みの初期化は自動微分の追跡対象外とする
        weight.normal_(0, std) # (0, std)の正規分布で上書き(in-place)
    return weight 

#### 実装のポイント
##### **1行目: fan_inの決定**
三項演算子で`fan_in`(入力数)を決めている
- `weight.dim() >= 2`(2次元以上)→`weight.shape[1]`=列数を使う
- そうではない(1次元)→`weight.shape[0]`=要素数を使う

なぜ2次元で`shape[1]`か：`nn.Linear`の重みは慣習的に`[out_features, in_features]`という形なので、
`shape[1]`が入力数`fan_in`にあたる

##### **2行目: 標準偏差の計算**
Kaiming初期化の式をそのまま実装


##### **3~4行目: 重みの書き換え**
- `weight.normal_(0, std)`\
`weight` の全要素を、平均0・標準偏差`std`の正規分布からサンプリングした値で上書きします。
末尾のアンダースコア `_` が重要で、これはPyTorchの **in-place(破壊的)操作**の印です。
新しいテンソルを作らず、`weight`自身の中身を直接書き換えます。
問題文の「Initialize weight in-place」の要求に対応する部分です。
(比較：`weight.normal()` のような非破壊版はなく、新規テンソルを作りたい場合は
 `torch.normal(...)` を使う。ここでは既存の `weight` を初期化したいので in-place版が適切)
 
- `with torch.no_grad()`:
この `with` ブロック内では、PyTorchが自動微分の追跡を停止します。\
なぜ必要か：`weight` は学習対象パラメータ(`requires_grad=True`)であることが多い。
追跡を止めずに `normal_` で書き換えると、
  - 「初期化という操作」が計算グラフに記録されてしまう
  - 場合によっては `RuntimeError`(in-place操作が勾配計算と衝突)になる

初期化は「学習とは無関係に値をセットするだけ」の操作なので、
勾配追跡から切り離す必要があります。
`torch.no_grad()` がそのためのブロックです。重みの初期化や手動更新では定型的に使われます。

In [11]:
# 🧪 Debug
import math
w = torch.empty(256, 512)
kaiming_init(w)
print(f'Mean: {w.mean():.4f} (expect ~0)')
print(f'Std:  {w.std():.4f} (expect {math.sqrt(2/512):.4f})')

Mean: 0.0000 (expect ~0)
Std:  0.0624 (expect 0.0625)


In [5]:
# ✅ SUBMIT
from torch_judge import check
check('weight_init')


🧪 Testing: Kaiming Initialization (Easy)
──────────────────────────────────────────────────
  ✅ [1/4] Mean approximately 0 (1.8ms)
  ✅ [2/4] Std matches sqrt(2/fan_in) (1.7ms)
  ✅ [3/4] Returns same tensor (in-place) (0.1ms)
  ✅ [4/4] Smaller fan_in gives larger std (0.2ms)
──────────────────────────────────────────────────
  🎉 All 4 tests passed! (3.8ms total)
  Progress saved. Run status() to see your dashboard.

